In [1]:
import os, re, json, requests
import pandas as pd
from datetime import datetime, timedelta, date as dt_date, time as dtime
from zoneinfo import ZoneInfo

# ══════════════════════════════════════════════════════════════════════════════
# NO-COVER PERIOD
# ══════════════════════════════════════════════════════════════════════════════
NO_COVER_START_DATE = dt_date(2026, 8, 29)
NO_COVER_START_HOUR = 6
NO_COVER_START_MIN  = 0
NO_COVER_END_DATE   = dt_date(2026, 8, 29)
NO_COVER_END_HOUR   = 21
NO_COVER_END_MIN    = 0

MY_EMAIL  = "huuchinh.nguyen@concentrix.com"

EMAIL_TO = (
    "atul.pathak@concentrix.com;"
    "rajat.roy@concentrix.com;"
    "puneet.suneja@concentrix.com;"
    "kirpan.patar@concentrix.com;"
    "ML.HOC.Expedia.Hierarchy@concentrix.com;"
    "EG_CAI_RAYAH_expedia_global_rtm@concentrix.com"
)

EMAIL_CC = (
    "urmila.chakka1@concentrix.com;"
    "van.tran@concentrix.com;"
    "duonghoangvu.pham@concentrix.com"
)

TEAMS_WEBHOOK_RTA = (
    "https://default599e51d62f8c43478e591f795a51a9.8c.environment.api.powerplatform.com:443"
    "/powerautomate/automations/direct/workflows/30e46f2733bc4e48a92dc32f90ba9329"
    "/triggers/manual/paths/invoke?api-version=1&sp=%2Ftriggers%2Fmanual%2Frun"
    "&sv=1.0&sig=xkD_H8_VvQh_XzhybXDxV3_gWFyC0E4-3Bpe_MJDJ44"
)
first_glob    = r"C:\Users\huuchinh.nguyen"
HC_PARQUET    = (
    f"{first_glob}/Concentrix Corporation"
    "/WFM-Expedia-HCM - Branding files"
    "/BI_Task/CODE/Resources/hc_extend_combination.parquet"
)
SCHEDULE_FILE = (
    r"C:\Users\huuchinh.nguyen\Concentrix Corporation"
    r"\WFM-Expedia-HCM - Branding files\Schedule\Schedule (Ops version)"
    r"\2026\Master_Schedule_Merged.xlsx"
)

TZ_VNT             = ZoneInfo("Asia/Ho_Chi_Minh")
TZ_PST             = ZoneInfo("America/Los_Angeles")
TZ_IST             = ZoneInfo("Asia/Kolkata")
LEAVE_CODES        = {'AL','LWP','CO','SL','EL','ML','PL','SPL','BL','CL','HO'}
EXCLUDE_SHIFTS     = {'OFF','TERMINATION','TERM','RESIGNED'}
WO_CODES           = {'WO'}
TL_DESIG_KEYWORDS  = ["Team Leader", "Operations"]
FORCE_TRIGGER      = True
SEND_EMAIL         = True
SHIFT_TYPES        = ['Morning', 'Afternoon', 'Night']
SHIFT_LABEL        = {
    'Morning':   'Morning Shift',
    'Afternoon': 'Afternoon Shift',
    'Night':     'Night Shift',
}


# ── Time helpers ───────────────────────────────────────────────────────────────
def _fmt_time(dt):
    h  = dt.hour % 12 or 12
    ap = "AM" if dt.hour < 12 else "PM"
    return f"{h}:{dt.minute:02d} {ap}"

def _d_label(d):
    return f"{d.strftime('%B')} {d.day}"

def parse_shift_times(shift_str):
    try:
        p = str(shift_str).strip().split('-')
        if len(p) == 2:
            s, e = p[0].strip(), p[1].strip()
            if len(s) == 4 and len(e) == 4 and s.isdigit() and e.isdigit():
                return dtime(int(s[:2]), int(s[2:])), dtime(int(e[:2]), int(e[2:]))
    except:
        pass
    return None, None

def is_overnight(shift_str):
    s, e = parse_shift_times(shift_str)
    return s is not None and e is not None and e.hour <= 8 and s.hour >= 18

def classify_shift(shift_str):
    s, _ = parse_shift_times(shift_str)
    if s is None:
        return None
    h = s.hour
    if 5 <= h < 10:  return 'Morning'
    if 10 <= h < 20: return 'Afternoon'
    return 'Night'

def is_working_shift(shift):
    u = str(shift).strip().upper()
    return (u not in EXCLUDE_SHIFTS and u not in LEAVE_CODES
            and u not in WO_CODES and u not in {'', 'NAN', 'NONE'}
            and '-' in u)

def shift_multizone_str(shift_str, sched_date):
    s_t, e_t = parse_shift_times(shift_str)
    if s_t is None:
        return shift_str
    overnight = is_overnight(shift_str)
    e_date    = sched_date + timedelta(days=1) if overnight else sched_date
    s_dt = datetime(sched_date.year, sched_date.month, sched_date.day,
                    s_t.hour, s_t.minute, tzinfo=TZ_VNT)
    e_dt = datetime(e_date.year, e_date.month, e_date.day,
                    e_t.hour, e_t.minute, tzinfo=TZ_VNT)
    s_pst   = s_dt.astimezone(TZ_PST)
    e_pst   = e_dt.astimezone(TZ_PST)
    s_ist   = s_dt.astimezone(TZ_IST)
    e_ist   = e_dt.astimezone(TZ_IST)
    pst_lbl = "PDT" if s_pst.utcoffset().total_seconds() == -7*3600 else "PST"
    if overnight:
        vnt_part = (f"{_d_label(sched_date)}, {_fmt_time(s_dt)}"
                    f" – {_d_label(e_date)}, {_fmt_time(e_dt)} VNT")
    else:
        vnt_part = (f"{_d_label(sched_date)}, {_fmt_time(s_dt)}"
                    f" – {_fmt_time(e_dt)} VNT")
    pst_part = f"{_fmt_time(s_pst)} – {_fmt_time(e_pst)} {pst_lbl}"
    ist_part = f"{_fmt_time(s_ist)} – {_fmt_time(e_ist)} IST"
    return f"{vnt_part} ({pst_part} / {ist_part})"


# ── Cover period datetime objects ─────────────────────────────────────────────
cover_start_dt = datetime.combine(
    NO_COVER_START_DATE, dtime(NO_COVER_START_HOUR, NO_COVER_START_MIN))
cover_end_dt   = datetime.combine(
    NO_COVER_END_DATE,   dtime(NO_COVER_END_HOUR,   NO_COVER_END_MIN))
n_days         = (NO_COVER_END_DATE - NO_COVER_START_DATE).days + 1
cover_dates    = [NO_COVER_START_DATE + timedelta(days=i) for i in range(n_days)]

cover_start_vnt = cover_start_dt.replace(tzinfo=TZ_VNT).astimezone(TZ_VNT).replace(tzinfo=None)
cover_start_pst = datetime.combine(NO_COVER_START_DATE,
    dtime(NO_COVER_START_HOUR, NO_COVER_START_MIN), tzinfo=TZ_VNT).astimezone(TZ_PST)
cover_start_ist = datetime.combine(NO_COVER_START_DATE,
    dtime(NO_COVER_START_HOUR, NO_COVER_START_MIN), tzinfo=TZ_VNT).astimezone(TZ_IST)
cover_end_pst   = datetime.combine(NO_COVER_END_DATE,
    dtime(NO_COVER_END_HOUR, NO_COVER_END_MIN), tzinfo=TZ_VNT).astimezone(TZ_PST)
cover_end_ist   = datetime.combine(NO_COVER_END_DATE,
    dtime(NO_COVER_END_HOUR, NO_COVER_END_MIN), tzinfo=TZ_VNT).astimezone(TZ_IST)
pst_lbl = "PDT" if cover_start_pst.utcoffset().total_seconds() == -7*3600 else "PST"

print(f"[COVER PERIOD] {_d_label(NO_COVER_START_DATE)} "
      f"{NO_COVER_START_HOUR:02d}:{NO_COVER_START_MIN:02d} VNT"
      f" → {_d_label(NO_COVER_END_DATE)} "
      f"{NO_COVER_END_HOUR:02d}:{NO_COVER_END_MIN:02d} VNT"
      f" ({n_days} day(s), dates: {[str(d) for d in cover_dates]})")


# ── Load Master Schedule ───────────────────────────────────────────────────────
print("\n[LOAD] Master Schedule...")
sched_raw = pd.read_excel(SCHEDULE_FILE, dtype=str)
sched_raw.columns = [str(c).strip() for c in sched_raw.columns]
for old, new in [('Email Id','Email'),('EMAIL','Email'),('email','Email')]:
    if old in sched_raw.columns and new not in sched_raw.columns:
        sched_raw.rename(columns={old: new}, inplace=True)
date_cols = {c: c[:10] for c in sched_raw.columns
             if re.match(r'^\d{4}-\d{2}-\d{2}', c)}
sched_raw.rename(columns=date_cols, inplace=True)
sched_raw['email_key'] = (sched_raw.get('Email', pd.Series(dtype=str))
                          .fillna('').astype(str).str.strip().str.lower())
oracle_col = next((c for c in ['OracleID','Emp ID','Oracle ID','EMP ID']
                   if c in sched_raw.columns), None)
if oracle_col:
    sched_raw[oracle_col] = (sched_raw[oracle_col].astype(str)
                             .str.strip().str.split(".").str[0])
print(f"  Rows: {len(sched_raw)} | OracleID col: '{oracle_col}'")


# ── Load HC Parquet — Designation filter ──────────────────────────────────────
print("[LOAD] HC Parquet...")
tl_oracle_ids = set()
desig_map     = {}
try:
    import polars as pl
    hc_pl     = pl.read_parquet(HC_PARQUET)
    latest_dt = hc_pl["Date"].max()
    desig_filter = None
    for kw in TL_DESIG_KEYWORDS:
        f = pl.col("Designation").str.contains(kw)
        desig_filter = f if desig_filter is None else (desig_filter | f)
    hc_filt = (
        hc_pl
        .filter(pl.col("Date") == latest_dt)
        .filter(desig_filter)
        .select([c for c in ["OracleID","Designation"] if c in hc_pl.columns])
        .unique(subset=["OracleID"])
        .to_pandas()
    )
    hc_filt["OracleID"] = (hc_filt["OracleID"].astype(str)
                           .str.strip().str.split(".").str[0])
    tl_oracle_ids = set(hc_filt["OracleID"].tolist())
    desig_map     = dict(zip(hc_filt["OracleID"], hc_filt["Designation"]))
    print(f"  TL/Ops agents: {len(tl_oracle_ids)} | Parquet date: {latest_dt}")
except Exception as e:
    print(f"  [WARN] HC Parquet load failed: {e}")


# ── Check my own shift → trigger ──────────────────────────────────────────────
today_str = str(dt_date.today())
my_shift  = 'UNKNOWN'
if today_str in sched_raw.columns:
    my_row = sched_raw[sched_raw['email_key'] == MY_EMAIL.lower()]
    if not my_row.empty:
        my_shift = str(my_row.iloc[0][today_str]).strip().upper()
is_wo = FORCE_TRIGGER or (my_shift in WO_CODES)
print(f"\n[MY SCHEDULE] Today → '{my_shift}' | Trigger: {'YES ✅' if is_wo else 'NO ❌'}")

if not is_wo:
    print("RTA is on the floor — no notification needed.")

else:
    # ── Melt schedule for all dates in cover window ───────────────────────────
    avail_date_cols = [str(d) for d in cover_dates if str(d) in sched_raw.columns]
    missing_dates   = [str(d) for d in cover_dates if str(d) not in sched_raw.columns]
    if missing_dates:
        print(f"  [WARN] Dates not in schedule: {missing_dates}")

    id_cols = [c for c in ['email_key','Email','Employee Name', oracle_col]
               if c and c in sched_raw.columns]
    if avail_date_cols:
        melted = (sched_raw[id_cols + avail_date_cols]
                  .melt(id_vars=id_cols,
                        value_vars=avail_date_cols,
                        var_name='sched_date_str',
                        value_name='shift'))
        melted['sched_date'] = pd.to_datetime(melted['sched_date_str']).dt.date
        melted['shift']      = melted['shift'].fillna('').astype(str).str.strip()
        melted['shift_type'] = melted['shift'].apply(classify_shift)
        melted = melted[melted['shift'].apply(is_working_shift)].copy()

        if oracle_col and tl_oracle_ids:
            sched_tl = melted[melted[oracle_col].isin(tl_oracle_ids)].copy()
        else:
            print("  [WARN] Falling back to all agents (no OracleID filter)")
            sched_tl = melted.copy()

        sched_tl['Designation'] = (sched_tl[oracle_col].map(desig_map)
                                   if oracle_col else 'Unknown')

        def _get_shift_start_dt(row):
            s, _ = parse_shift_times(row['shift'])
            if s is None:
                return pd.NaT
            return datetime.combine(row['sched_date'], s)

        sched_tl['shift_start_dt'] = sched_tl.apply(_get_shift_start_dt, axis=1)

        def _get_shift_end_dt(row):
            _, e = parse_shift_times(row['shift'])
            if e is None:
                return pd.NaT
            overnight = is_overnight(row['shift'])
            e_date    = row['sched_date'] + timedelta(days=1) if overnight else row['sched_date']
            return datetime.combine(e_date, e)

        sched_tl['shift_end_dt'] = sched_tl.apply(_get_shift_end_dt, axis=1)
        sched_tl = sched_tl.dropna(subset=['shift_start_dt', 'shift_end_dt'])

        sched_tl = sched_tl[
            (sched_tl['shift_start_dt'] < cover_end_dt) &
            (sched_tl['shift_end_dt']   > cover_start_dt)
        ].copy()

        display_name_col = next(
            (c for c in ['Employee Name','Name'] if c in sched_tl.columns), None)
        sched_tl['display_name']  = (sched_tl[display_name_col].fillna(sched_tl['email_key'])
                                     if display_name_col else sched_tl['email_key'])
        sched_tl['display_email'] = (sched_tl['Email'].fillna(sched_tl['email_key'])
                                     if 'Email' in sched_tl.columns else sched_tl['email_key'])

        print(f"\n[TL SHIFTS] Active TLs/Ops in cover window: {len(sched_tl)}")
        for _, r in sched_tl.sort_values(['sched_date','shift_start_dt']).iterrows():
            print(f"  {str(r['display_name']):<28} "
                  f"{str(r['sched_date'])} {r['shift']:<12} "
                  f"{r['shift_type']:<12} {r.get('Designation','')}")
    else:
        sched_tl = pd.DataFrame()
        print("  [WARN] No matching date columns found in schedule")

    # ── Build coverage dict: {date_str: {shift_type: [emails]}} ──────────────
    coverage = {}
    coverage_names = {}
    for d in cover_dates:
        d_str = str(d)
        coverage[d_str]       = {st: [] for st in SHIFT_TYPES}
        coverage_names[d_str] = {st: [] for st in SHIFT_TYPES}
        if sched_tl.empty:
            continue
        grp_d = sched_tl[sched_tl['sched_date'] == d]
        for stype in SHIFT_TYPES:
            grp_s = grp_d[grp_d['shift_type'] == stype]
            if not grp_s.empty:
                coverage[d_str][stype] = grp_s['display_email'].tolist()
                coverage_names[d_str][stype] = grp_s['display_name'].tolist()

    # ── Coverage table ────────────────────────────────────────────────────────
    table_data = {}
    for d in cover_dates:
        d_str   = str(d)
        col_lbl = f"{d.strftime('%b')} {d.day}"
        table_data[col_lbl] = {}
        for stype in SHIFT_TYPES:
            emails = coverage[d_str].get(stype, [])
            table_data[col_lbl][stype] = (
                '\n'.join(e.split('@')[0] for e in emails) if emails else '—')
    coverage_df = (pd.DataFrame(table_data)
                   .reindex(SHIFT_TYPES)
                   .rename_axis("Shift Type"))

    print(f"\n{'═'*65}")
    print("COVERAGE TABLE (Team Leaders on schedule)")
    print(f"{'═'*65}")
    print(coverage_df.to_string())

    # ── Build email body ──────────────────────────────────────────────────────
    body = [
        "Hi All,",
        "",
        f"Please be informed that there will be no RTA coverage from VNM "
        f"starting {_d_label(NO_COVER_START_DATE)} at "
        f"{NO_COVER_START_HOUR:02d}:{NO_COVER_START_MIN:02d} VNT "
        f"({_fmt_time(cover_start_pst)} {pst_lbl} / {_fmt_time(cover_start_ist)} IST) "
        f"until {_d_label(NO_COVER_END_DATE)} at "
        f"{NO_COVER_END_HOUR:02d}:{NO_COVER_END_MIN:02d} VNT "
        f"({_fmt_time(cover_end_pst)} {pst_lbl} / {_fmt_time(cover_end_ist)} IST).",
        "",
        "We kindly request the Operations team to assist with Queue management, "
        "IC, and Attendance tracking during this period.",
        "",
        "For assistance regarding LG Chat, please contact "
        "the following Team Leaders based on the shifts below:",
    ]
    has_any = False
    for d in cover_dates:
        d_str     = str(d)
        d_has     = any(coverage[d_str].get(st) for st in SHIFT_TYPES)
        if not d_has:
            continue
        has_any = True
        body.append("")
        if n_days > 1:
            body.append(f"── {_d_label(d)} ──")
            body.append("")
        for stype in SHIFT_TYPES:
            emails = coverage[d_str].get(stype, [])
            if not emails:
                continue
            grp_d_s = (sched_tl[(sched_tl['sched_date'] == d) &
                                  (sched_tl['shift_type'] == stype)]
                       if not sched_tl.empty else pd.DataFrame())
            if not grp_d_s.empty:
                time_range = shift_multizone_str(grp_d_s['shift'].iloc[0], d)
            else:
                time_range = stype
            body.append(f"{SHIFT_LABEL[stype]} / {time_range}:")
            for em in emails:
                body.append(f"\u2022 {em.strip()}")
            body.append("")
    if not has_any:
        body += [
            "",
            "Note: No Team Leaders are currently scheduled during this period. "
            "Please coordinate directly with the Operations team.",
            "",
        ]
    body += ["Thank you for your support and cooperation.", "", "Best regards"]
    email_body = "\n".join(body)

        # ── Build HTML email body (table format) ──────────────────────────────────
    th_style = ('bgcolor="#1e3a5f" style="color:#ffffff;padding:8px 12px;'
                'border:1px solid #2c4f7c;text-align:left;font-family:'
                'Segoe UI,Arial,sans-serif;font-size:12px;"')
    td_label = ('style="padding:7px 12px;border:1px solid #ddd;'
                'font-family:Segoe UI,Arial,sans-serif;font-size:12px;'
                'font-weight:bold;background:#f5f5f5;vertical-align:top;"')
    td_cell  = ('style="padding:7px 12px;border:1px solid #ddd;'
                'font-family:Segoe UI,Arial,sans-serif;font-size:12px;'
                'vertical-align:top;"')

    date_headers = "".join(
        f'<th {th_style}>{_d_label(d)}</th>' for d in cover_dates)
    table_rows = ""
    for stype in SHIFT_TYPES:
        cells = ""
        for d in cover_dates:
            emails = coverage[str(d)].get(stype, [])
            if emails:
                cell_content = "<br>".join(f"&bull; {e.strip()}" for e in emails)
            else:
                cell_content = "&#8212;"
            rbg = "#f9f9f9" if cover_dates.index(d) % 2 == 0 else "#ffffff"
            cells += (f'<td {td_cell} bgcolor="{rbg}" '
                      f'style="padding:7px 12px;border:1px solid #ddd;'
                      f'font-family:Segoe UI,Arial,sans-serif;font-size:12px;'
                      f'vertical-align:top;background-color:{rbg};">'
                      f'{cell_content}</td>')
        table_rows += (f'<tr><td {td_label}>{SHIFT_LABEL[stype]}</td>'
                       f'{cells}</tr>')

    coverage_table_html = (
        f'<table border="1" cellpadding="0" cellspacing="0" '
        f'style="border-collapse:collapse;font-size:12px;'
        f'font-family:Segoe UI,Arial,sans-serif;">'
        f'<thead><tr><th {th_style}>Shift / Date</th>{date_headers}</tr></thead>'
        f'<tbody>{table_rows}</tbody></table>'
    )

    intro_lines = [
        "Hi All,",
        "",
        f"Please be informed that there will be no RTA coverage from VNM "
        f"starting {_d_label(NO_COVER_START_DATE)} at "
        f"{NO_COVER_START_HOUR:02d}:{NO_COVER_START_MIN:02d} VNT "
        f"({_fmt_time(cover_start_pst)} {pst_lbl} / {_fmt_time(cover_start_ist)} IST) "
        f"until {_d_label(NO_COVER_END_DATE)} at "
        f"{NO_COVER_END_HOUR:02d}:{NO_COVER_END_MIN:02d} VNT "
        f"({_fmt_time(cover_end_pst)} {pst_lbl} / {_fmt_time(cover_end_ist)} IST).",
        "",
        "We kindly request the Operations team to assist with Queue management, "
        "IC, and Attendance tracking during this period.",
        "",
        "For assistance regarding LG Chat, please contact "
        "the following Team Leaders based on the shifts below:",
    ]
    closing_lines = [
        "",
        "Thank you for your support and cooperation.",
        "",
        "Best regards",
    ]

    def _to_html_p(lines):
        return "".join(
            f"<p style='margin:4px 0;font-family:Segoe UI,Arial,sans-serif;"
            f"font-size:13px;'>{l if l else '&nbsp;'}</p>"
            for l in lines)

    email_html_body = (
        f'<html><body style="font-family:Segoe UI,Arial,sans-serif;">'
        f'{_to_html_p(intro_lines)}'
        f'<br>{coverage_table_html}<br>'
        f'{_to_html_p(closing_lines)}'
        f'</body></html>'
    )

    # ── Build Teams message (plain text, same as email) ───────────────────────
    teams_html = (
        email_body
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace("\n", "<br>")
    )

    # ── Subject ───────────────────────────────────────────────────────────────
    if NO_COVER_START_DATE == NO_COVER_END_DATE:
        subject = (f"Expedia - RTA Coverage Update: "
                   f"No RTA from VNM ({_d_label(NO_COVER_START_DATE)})")
    else:
        subject = (f"Expedia - RTA Coverage Update: "
                   f"No RTA from VNM ({_d_label(NO_COVER_START_DATE)}"
                   f" - {_d_label(NO_COVER_END_DATE)})")

    # ── Function 1: Send email via Outlook ────────────────────────────────────
    def send_outlook_email():
        if not SEND_EMAIL:
            print("[OUTLOOK] Skipped (SEND_EMAIL = False)")
            return
        try:
            import win32com.client as win32
            outlook       = win32.Dispatch('outlook.application')
            mail          = outlook.CreateItem(0)
            mail.To       = EMAIL_TO
            mail.CC       = EMAIL_CC
            mail.Subject  = subject
            mail.HTMLBody = email_html_body
            mail.Send()
            print(f"[OUTLOOK] Email sent → {EMAIL_TO}")
        except ImportError:
            print("[OUTLOOK] win32com not available — run: pip install pywin32")
        except Exception as e:
            print(f"[OUTLOOK] Error: {e}")

    # ── Function 2: Send Teams webhook ────────────────────────────────────────
    def send_teams_rta():
        try:
            r = requests.post(
                TEAMS_WEBHOOK_RTA,
                headers={"Content-Type": "application/json"},
                data=json.dumps({"html": email_html_body}),
                timeout=30
            )
            print(f"[TEAMS] {'Sent ✅' if r.status_code in (200,202) else f'Failed [{r.status_code}]: {r.text[:200]}'}")
        except Exception as e:
            print(f"[TEAMS] Error: {e}")

    # ── Console preview ────────────────────────────────────────────────────────
    print(f"\n{'═'*65}")
    print("EMAIL PREVIEW")
    print(f"{'═'*65}")
    print(f"Subject : {subject}")
    print(f"To      : {EMAIL_TO}")
    print(f"CC      : {EMAIL_CC}")
    print(f"{'─'*65}")
    print(email_body)
    print(f"{'─'*65}")

    send_outlook_email()
    send_teams_rta()

[COVER PERIOD] August 29 06:00 VNT → August 29 21:00 VNT (1 day(s), dates: ['2026-08-29'])

[LOAD] Master Schedule...
  Rows: 203 | OracleID col: 'OracleID'
[LOAD] HC Parquet...
  TL/Ops agents: 7 | Parquet date: 2026-11-08

[MY SCHEDULE] Today → '2100-0600' | Trigger: YES ✅

[TL SHIFTS] Active TLs/Ops in cover window: 3
  TRAN HOANG MY ANH            2026-08-29 0500-1400    Morning      SME, Operations
  CHAU THIEN KIM               2026-08-29 0600-1500    Morning      Team Leader, Operations
  TRAN THI NGOC THAO           2026-08-29 0900-1800    Morning      Team Leader, Operations

═════════════════════════════════════════════════════════════════
COVERAGE TABLE (Team Leaders on schedule)
═════════════════════════════════════════════════════════════════
                                                      Aug 29
Shift Type                                                  
Morning     hoangmyanh.tran\nthingocthao.tran\nthienkim.chau
Afternoon                                          